# ChromaDB Quick Start

**Chroma** is open-source data infrastructure for AI retrieval.

In this notebook you will:
- Store text with metadata in a local in-memory collection
- Query with dense (semantic), lexical (full-text), and hybrid search
- Use metadata for filtering and LLM context

## Setup

Chroma's default embedding model (`all-MiniLM-L6-v2`) downloads and runs **locally** — no API keys required.

Docs: [chromadb](https://docs.trychroma.com/)

In [1]:
import chromadb  # https://docs.trychroma.com/

## Chroma Client

We use Chroma's [in-memory client](https://docs.trychroma.com/docs/run-chroma/clients#in-memory-client):
- Starts a server inside your Python process
- Data disappears when the process exits

For persistence, see the [persistent client](https://docs.trychroma.com/docs/run-chroma/clients#persistent-client) or [client-server mode](https://docs.trychroma.com/docs/run-chroma/client-server).

In [2]:
# Ephemeral client — fine for learning and quick experiments
client = chromadb.Client()

# One named index inside this client
collection = client.get_or_create_collection(
    name="test-collection",
)

## Ingestion of Data and Metadata

In [3]:
documents = [
    "The Eiffel Tower is located in Paris.",
    "The Great Wall of China can be seen from space.",
    "The Colosseum is an ancient Roman gladiatorial arena.",
    "Mount Everest is the tallest mountain in the world.",
]
metadatas = [
    {"city": "Paris", "type": "monument"},
    {"country": "China", "type": "landmark"},
    {"city": "Rome", "type": "arena"},
    {"location": "Nepal", "type": "mountain"},
]
ids = ["doc1", "doc2", "doc3", "doc4"]

collection.add(
    documents=documents,  # raw text Chroma will embed
    metadatas=metadatas,  # filterable key/value fields
    ids=ids,              # stable record identifiers
)

print("=== Data Ingested ===")
for doc_id, doc, meta in zip(ids, documents, metadatas):
    print(f"{doc_id}: {doc} | {meta}")

=== Data Ingested ===
doc1: The Eiffel Tower is located in Paris. | {'city': 'Paris', 'type': 'monument'}
doc2: The Great Wall of China can be seen from space. | {'country': 'China', 'type': 'landmark'}
doc3: The Colosseum is an ancient Roman gladiatorial arena. | {'city': 'Rome', 'type': 'arena'}
doc4: Mount Everest is the tallest mountain in the world. | {'location': 'Nepal', 'type': 'mountain'}


On first use, Chroma downloads `all-MiniLM-L6-v2` (~79 MB) and embeds each document automatically.

## Querying the Database

In [4]:
query = "Where is the Eiffel Tower?"

print("=== Query ===")
print(query)

results = collection.query(
    query_texts=[query],  # embed this string, then nearest-neighbor search
    n_results=2,          # return the 2 closest documents
)

print("\n=== Results ===")
for rank, (doc_id, doc, meta) in enumerate(
    zip(results["ids"][0], results["documents"][0], results["metadatas"][0]),
    start=1,
):
    print(f"Result {rank}: {doc_id}\n  Text: {doc}\n  Metadata: {meta}")

=== Query ===
Where is the Eiffel Tower?

=== Results ===
Result 1: doc1
  Text: The Eiffel Tower is located in Paris.
  Metadata: {'type': 'monument', 'city': 'Paris'}
Result 2: doc2
  Text: The Great Wall of China can be seen from space.
  Metadata: {'country': 'China', 'type': 'landmark'}


## Search Modalities

Chroma supports several retrieval styles:
- **Dense** — meaning-based (embeddings)
- **Lexical** — token/substring-based (full-text filters)
- **Hybrid** — combine both rank lists

### 1. Dense Search

**Dense search** uses embeddings: vectors that encode *meaning*.

Our four documents become points in a high-dimensional space. A query lands in the same space; nearby points are semantically similar chunks.

![](../assets/vector_db_query.png){height=512}

When you query `"where's the tallest building?"`, the embedding model projects that text into the shared space.

You can also **filter by metadata** — for example `{type: monument}` — before or alongside vector search.

##### Multi-modality

Embedding models can encode images, audio, and video too — not only text.

#### When to Use?

Dense search excels when:
- The query paraphrases the source (`"how do I return a product"` vs `"return policy"`)
- Exact keywords may be missing from the chunk

It struggles with:
- SKUs, legal citations, or rare jargon that never appeared in training data

#### How to Use?

Semantic search is enabled by default (`all-MiniLM-L6-v2`).

Narrow results with a metadata filter via `where`:

In [5]:
filtered = collection.query(
    query_texts=["tall building"],
    n_results=2,
    where={"type": "monument"},  # only rows whose metadata matches
)

print("=== Dense + metadata filter ===")
for doc_id, doc, meta in zip(
    filtered["ids"][0],
    filtered["documents"][0],
    filtered["metadatas"][0],
):
    print(f"{doc_id}: {doc} | {meta}")

=== Dense + metadata filter ===
doc1: The Eiffel Tower is located in Paris. | {'type': 'monument', 'city': 'Paris'}


### 2. Lexical Search (Full-Text Search)

**Lexical search** matches exact tokens in stored document text.

#### When to Use?

Lexical search shines when you need precision:
- Product IDs like `SKU-4892-X`
- Drug names, legal citations, or model numbers

Tradeoff: it will not bridge synonyms — searching `"cancel"` won't match `"terminate"`.

#### How to Use?

Pass **`where_document`** to `collection.get()` or `collection.query()` to filter on document content.

Supported operators (see [full-text search docs](https://docs.trychroma.com/docs/querying-collections/full-text-search)):
- `$contains` / `$not_contains` — substring match
- `$regex` / `$not_regex` — [regular expression](https://regex101.com) patterns
- `$and` / `$or` — combine multiple document filters

Full-text search is **case-sensitive**.

#### `$contains`

In [6]:
contains_hits = collection.get(
    where_document={"$contains": "Eiffel"},
    include=["documents", "metadatas"],
)

print("=== $contains: 'Eiffel' ===")
for doc_id, doc, meta in zip(
    contains_hits["ids"],
    contains_hits["documents"],
    contains_hits["metadatas"],
):
    print(f"{doc_id}: {doc} | {meta}")

=== $contains: 'Eiffel' ===
doc1: The Eiffel Tower is located in Paris. | {'type': 'monument', 'city': 'Paris'}


#### `$not_contains`

In [7]:
excluded = collection.get(
    where_document={"$not_contains": "Tower"},
    include=["documents"],
)

print("=== $not_contains: 'Tower' ===")
for doc_id, doc in zip(excluded["ids"], excluded["documents"]):
    print(f"{doc_id}: {doc}")

=== $not_contains: 'Tower' ===
doc2: The Great Wall of China can be seen from space.
doc3: The Colosseum is an ancient Roman gladiatorial arena.
doc4: Mount Everest is the tallest mountain in the world.


#### `$regex`

In [8]:
regex_hits = collection.get(
    where_document={"$regex": "Roman|China"},
    include=["documents", "metadatas"],
)

print("=== $regex: Roman|China ===")
for doc_id, doc, meta in zip(
    regex_hits["ids"],
    regex_hits["documents"],
    regex_hits["metadatas"],
):
    print(f"{doc_id}: {doc} | {meta}")

=== $regex: Roman|China ===
doc2: The Great Wall of China can be seen from space. | {'type': 'landmark', 'country': 'China'}
doc3: The Colosseum is an ancient Roman gladiatorial arena. | {'type': 'arena', 'city': 'Rome'}


#### Logical Operators

- `$and` — document must match **every** filter
- `$or` — document may match **any** filter

In [9]:
and_hits = collection.get(
    where_document={
        "$and": [
            {"$contains": "is"},           # verb appears in the sentence
            {"$regex": "Paris|Roman"},     # and mentions Paris or Roman
        ]
    },
    include=["documents"],
)

print("=== $and ===")
for doc_id, doc in zip(and_hits["ids"], and_hits["documents"]):
    print(f"{doc_id}: {doc}")

or_hits = collection.get(
    where_document={
        "$or": [
            {"$contains": "Everest"},
            {"$contains": "Eiffel"},
        ]
    },
    include=["documents"],
)

print("\n=== $or ===")
for doc_id, doc in zip(or_hits["ids"], or_hits["documents"]):
    print(f"{doc_id}: {doc}")

=== $and ===
doc1: The Eiffel Tower is located in Paris.
doc3: The Colosseum is an ancient Roman gladiatorial arena.

=== $or ===
doc1: The Eiffel Tower is located in Paris.
doc4: Mount Everest is the tallest mountain in the world.


#### Combining with Metadata Filtering

`query()` accepts both `where` (metadata) and `where_document` (text) in the same call.

In [10]:
combined = collection.query(
    query_texts=["famous landmark"],
    n_results=2,
    where={"type": "monument"},              # metadata gate
    where_document={"$contains": "Paris"},   # text gate
)

print("=== Dense + metadata + full-text filter ===")
for doc_id, doc, meta in zip(
    combined["ids"][0],
    combined["documents"][0],
    combined["metadatas"][0],
):
    print(f"{doc_id}: {doc} | {meta}")

=== Dense + metadata + full-text filter ===
doc1: The Eiffel Tower is located in Paris. | {'city': 'Paris', 'type': 'monument'}


### 3. Hybrid Search

**Hybrid search** runs dense and lexical strategies, then merges their rank lists.

Chroma's cloud Search API supports this natively; locally we use `search_utils.hybrid_search` with **reciprocal rank fusion (RRF)**. See [Cormack et al. (2009)](https://plg.uwaterloo.ca/~gvcormac/cormacksigir09-rrf.pdf).

In [11]:
from search_utils import hybrid_search

query = "Eiffel Tower Paris"

result = hybrid_search(
    collection=collection,
    query=query,
    doc_ids=ids,
    documents=documents,
    weights=(0.7, 0.3),  # 70% dense, 30% lexical
    k=60,                  # RRF smoothing constant
)

id_to_doc = dict(zip(ids, documents))

print("Dense: ", result.dense_ranking)
print("Lexical:", result.lexical_ranking)
print("Hybrid (RRF):")
for rank, doc_id in enumerate(result.hybrid_ranking[:3], start=1):
    print(f"  {rank}. {doc_id}: {id_to_doc[doc_id]}")

Dense:  ['doc1', 'doc2', 'doc4', 'doc3']
Lexical: ['doc1', 'doc2', 'doc3', 'doc4']
Hybrid (RRF):
  1. doc1: The Eiffel Tower is located in Paris.
  2. doc2: The Great Wall of China can be seen from space.
  3. doc4: Mount Everest is the tallest mountain in the world.


## Metadata

Metadata attaches structured fields to each record. See [Look at Your Data — Metadata](https://docs.trychroma.com/guides/build/look-at-your-data#metadata).

It serves two purposes in RAG:

**1. Filtering at query time** — narrow results without relying on semantic similarity:
- Source type (FAQs vs legal disclaimers)
- Date, author, or department
- Access permissions (only chunks the user may see)

**2. Context for the LLM** — metadata is returned with search hits, so you can cite sources accurately (e.g. `"Q3 2024 Financial Report, page 12"` or `"authored by the legal team"`).

When designing a schema, ask:
- Which fields will you filter on at query time?
- Which fields help the LLM interpret each chunk?

#### Filter by Metadata

Use `where` with `collection.get()` when you only need metadata matches — no embedding search required.

In [12]:
city_hits = collection.get(
    where={"city": "Paris"},  # exact metadata match
    include=["documents", "metadatas"],
)

print("=== Metadata-only filter (city = Paris) ===")
for doc_id, doc, meta in zip(
    city_hits["ids"],
    city_hits["documents"],
    city_hits["metadatas"],
):
    print(f"{doc_id}: {doc} | {meta}")

=== Metadata-only filter (city = Paris) ===
doc1: The Eiffel Tower is located in Paris. | {'city': 'Paris', 'type': 'monument'}


#### Pass Metadata to the LLM

In a RAG pipeline, bundle each retrieved chunk with its metadata so the model can ground answers and cite sources.

In [13]:
rag_hits = collection.query(
    query_texts=["famous ancient site"],
    n_results=2,
    include=["documents", "metadatas"],
)

print("=== Context blocks for the LLM ===")
for doc_id, doc, meta in zip(
    rag_hits["ids"][0],
    rag_hits["documents"][0],
    rag_hits["metadatas"][0],
):
    # Turn metadata into a short citation line the model can reference
    place = meta.get("city") or meta.get("country") or meta.get("location", "unknown")
    citation = f"[{doc_id} · {meta['type']} · {place}]"
    print(citation)
    print(doc)
    print()

=== Context blocks for the LLM ===
[doc3 · arena · Rome]
The Colosseum is an ancient Roman gladiatorial arena.

[doc2 · landmark · China]
The Great Wall of China can be seen from space.



## Collections in Your Chroma Database

A collection indexes records with one embedding configuration.

**Use a single collection when:**

- All data shares the same embedding model
- You want one search surface and can filter by metadata

**Use multiple collections when:**

- Data types need different embedding models (e.g. text vs images)
- You have multi-tenant isolation requirements